## *BEL’s 280-Channel HD EEG*

**Link:** https://www.bel.company/bel-eeg-system-one

In [71]:
# IMPORTS

import mne
from mne.preprocessing import ICA
from mne_icalabel import label_components
import numpy as np
import os
from pathlib import Path
import warnings
from scipy.stats import median_abs_deviation
from typing import List, Tuple, Optional, Dict

# Suppress runtime warnings for cleaner output
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [72]:
# CHANNEL MAPPING
def parse_gpsc(filepath: Path) -> List[Tuple[str, float, float, float]]:
    """Parse BEL .gpsc file into list of (name, x, y, z)."""
    channels = []
    with open(filepath, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                try:
                    name = parts[0]
                    x, y, z = map(float, parts[1:4])
                    channels.append((name, x, y, z))
                except ValueError:
                    continue
    return channels

def create_montage_from_gpsc(
    channels: List[Tuple[str, float, float, float]],
    coord_frame: str = 'head'
) -> mne.channels.DigMontage:
    """Create MNE montage from parsed GPSC channels."""
    if not channels:
        raise ValueError("No valid channels provided.")
    
    gpsc_array = np.array([ch[1:4] for ch in channels])
    mean_pos = gpsc_array.mean(axis=0)
    
    ch_pos = {
        ch[0]: np.array([
            ch[1] - mean_pos[0],
            ch[2] - mean_pos[1],
            ch[3] - mean_pos[2]
        ]) / 1000.0  # Convert mm → meters
        for ch in channels
    }
    
    return mne.channels.make_dig_montage(
        ch_pos=ch_pos,
        nasion=ch_pos.get('FidNz'),
        lpa=ch_pos.get('FidT9'),
        rpa=ch_pos.get('FidT10'),
        coord_frame=coord_frame
    )

class BELStandardizer:
    """Convenience class to standardize BEL EEG data."""
    def __init__(self, gpsc_file: Path, rename_map: Optional[Dict[str, str]] = None):
        self.gpsc_file = Path(gpsc_file)
        if not self.gpsc_file.exists():
            raise FileNotFoundError(f"GPSC file not found: {self.gpsc_file}")
        self.rename_map = rename_map or {}

    def standardize(self, raw: mne.io.Raw) -> mne.io.Raw:
        raw = raw.copy()
        # 1. Rename channels
        if self.rename_map:
            existing_map = {old: new for old, new in self.rename_map.items() if old in raw.ch_names}
            if existing_map:
                raw.rename_channels(existing_map)
        # 2. Apply montage
        channels = parse_gpsc(self.gpsc_file)
        montage = create_montage_from_gpsc(channels)
        raw.set_montage(montage, on_missing='warn')
        return raw



In [73]:
# BAD CHANNEL DETECTION FUNCTION
def detect_bad_channels(
    raw: mne.io.Raw,
    mad_threshold: float = 10.0,
    min_amplitude_uv: float = 0.1
) -> List[str]:
    """Detect flat and noisy EEG channels using MAD and Amplitude."""
    raw_eeg = raw.copy().pick("eeg")
    data_uv = raw_eeg.get_data() * 1e6
    amplitude = np.ptp(data_uv, axis=1)
    variance = np.var(data_uv, axis=1)

    # 1. Detect Flat Channels
    flat_mask = amplitude < min_amplitude_uv
    flat_chs = [ch for ch, is_flat in zip(raw_eeg.ch_names, flat_mask) if is_flat]

    # 2. Detect Noisy Channels (MAD)
    noisy_mask = np.zeros(len(amplitude), dtype=bool)
    for feat in (variance, amplitude):
        mad = median_abs_deviation(feat, scale="normal", nan_policy="omit")
        if not np.isnan(mad) and mad > 1e-12:
            z = (feat - np.nanmedian(feat)) / mad
            noisy_mask |= z > mad_threshold
    noisy_chs = [ch for ch, is_noisy in zip(raw_eeg.ch_names, noisy_mask) if is_noisy]

    bad_chs = sorted(set(flat_chs + noisy_chs))
    print(f"Detected {len(bad_chs)} bad channels: {bad_chs}")
    return bad_chs

In [74]:
# LOAD DATA

# Paths
FILE_PATH = "./MISP_P001_EC_Trial1.mff"
GPSC_FILE = "./ghw280_from_egig.gpsc"



raw = mne.io.read_raw_egi(FILE_PATH, preload=True)




Reading EGI MFF Header from /mnt/movement/users/jaizor/xtra/lectures/eeg/MISP_P001_EC_Trial1.mff...
    Reading events ...
    Assembling measurement info ...
    Excluding events {} ...
Reading 0 ... 67625  =      0.000 ...   135.250 secs...


In [75]:
# BEL Renaming Map (EGI Numeric -> BEL Standard)
RENAME_MAP = {str(i): f'E{i}' for i in range(1, 281)}
RENAME_MAP['REF CZ'] = 'Cz'


# Apply BEL Standardization (Renaming + Montage)
standardizer = BELStandardizer(Path(GPSC_FILE), RENAME_MAP)
raw = standardizer.standardize(raw)
print("BEL Standardization complete.")

BEL Standardization complete.


In [76]:
# Bandpass Filter

LOW_CUTOFF = 1.0
HIGH_CUTOFF = 100.0
NOTCH_FREQ = 50.0  # Use 60.0 for US, 50.0 for EU


raw.filter(l_freq=LOW_CUTOFF, h_freq=HIGH_CUTOFF, picks='eeg', verbose=False)
# Notch Filter
raw.notch_filter(freqs=NOTCH_FREQ, picks='eeg', verbose=False)

# Drop Reference Channel (Cz)
if 'Cz' in raw.ch_names:
    raw.drop_channels(['Cz'])
    print("Dropped Cz reference channel.")



Dropped Cz reference channel.


In [77]:
# Detect Bad Channels
bad_channels = detect_bad_channels(raw, mad_threshold=15.0, min_amplitude_uv=0.1)

Detected 15 bad channels: ['E139', 'E146', 'E181', 'E186', 'E19', 'E2', 'E268', 'E270', 'E32', 'E43', 'E52', 'E53', 'E8', 'E88', 'E94']


In [78]:
# Mark and Interpolate 

if bad_channels:
    raw.info['bads'] = bad_channels
    raw.interpolate_bads(reset_bads=True)
    print("Bad channels interpolated successfully.")
else:
    print("No bad channels detected.")

Setting channel interpolation method to {'eeg': 'spline'}.
Interpolating bad channels.
    Automatic origin fit: head of radius 99.3 mm
Computing interpolation matrix from 265 sensor positions
Interpolating 15 sensors
Bad channels interpolated successfully.


In [79]:
# Average Reference
raw.set_eeg_reference('average', verbose=False)
print("Average reference applied.")

Average reference applied.


In [80]:
# ICLabel

ICA_N_COMPONENTS = 0.99  # Retain components explaining 95% variance
RANDOM_STATE = 99
ICLABEL_THRESHOLDS = {
    'eye blink': 0.70,
    'heart beat': 0.70,
    'muscle artifact': 0.70,
    'line noise': 0.70,
    'channel noise': 0.70
}

print("Fitting ICA decomposition...")
with warnings.catch_warnings():
    warnings.filterwarnings('ignore', category=RuntimeWarning)
    ica = ICA(
        n_components=ICA_N_COMPONENTS,
        method='picard',
        fit_params=dict(ortho=False, extended=True),
        random_state=RANDOM_STATE,
        max_iter='auto'
    )
    ica.fit(raw, picks='eeg')
print(f"ICA fitted with {ica.n_components_} components")

print("Classifying components with ICLabel...")
labels_dict = label_components(raw, ica, method='iclabel')

# Identify components to exclude based on ICLabel probabilities
exclude_idx = []
for i, (label, prob_vec) in enumerate(zip(labels_dict['labels'], labels_dict['y_pred_proba'])):
    label_key = label.lower().strip()
    if label_key in ICLABEL_THRESHOLDS and np.max(prob_vec) >= ICLABEL_THRESHOLDS[label_key]:
        exclude_idx.append(i)

ica.exclude = sorted(set(exclude_idx))

# Report exclusions
print(f"Excluding {len(ica.exclude)} components based on ICLabel:")
for idx in ica.exclude:
    label = labels_dict['labels'][idx]
    prob = np.max(labels_dict['y_pred_proba'][idx])
    print(f"  Component {idx:02d}: {label:<18} (probability: {prob:.2f})")

# Apply ICA
print("Applying ICA to reconstruct cleaned data...")
raw_clean = ica.apply(raw.copy())
print("ICA artifacts removed.")

Fitting ICA decomposition...
Fitting ICA to data using 280 channels (please be patient, this may take a while)
Selecting by explained variance: 73 components
Fitting ICA took 29.1s.
ICA fitted with 73 components
Classifying components with ICLabel...
Excluding 7 components based on ICLabel:
  Component 03: eye blink          (probability: 0.90)
  Component 10: channel noise      (probability: 0.90)
  Component 13: eye blink          (probability: 0.79)
  Component 42: eye blink          (probability: 0.87)
  Component 54: channel noise      (probability: 0.94)
  Component 58: channel noise      (probability: 0.75)
  Component 70: channel noise      (probability: 0.83)
Applying ICA to reconstruct cleaned data...
Applying ICA to Raw instance
    Transforming to ICA space (73 components)
    Zeroing out 7 ICA components
    Projecting back using 280 PCA components
ICA artifacts removed.


In [81]:
raw_clean

<RawMff | signal1.bin, 280 x 67626 (135.3 s), ~144.8 MiB, data loaded>

In [82]:
# Save Data
OUTPUT_DIR = "./clean_data/"

os.makedirs(OUTPUT_DIR, exist_ok=True)
out_name = Path(FILE_PATH).stem + "_clean.fif"
save_path = os.path.join(OUTPUT_DIR, out_name)

raw_clean.save(save_path, overwrite=True)
print(f"Saved cleaned data to: {save_path}")


Writing /mnt/movement/users/jaizor/xtra/lectures/eeg/clean_data/MISP_P001_EC_Trial1_clean.fif


Closing /mnt/movement/users/jaizor/xtra/lectures/eeg/clean_data/MISP_P001_EC_Trial1_clean.fif
[done]
Saved cleaned data to: ./clean_data/MISP_P001_EC_Trial1_clean.fif
